# Pokemon TCG AI Battle — Agent Testing

Run inside a Kaggle competition notebook (cabt is pre-installed there).

1. Set `GITHUB_URL`.
2. Edit `AGENTS`, `NICKNAMES`, and `N_GAMES` in section 4.
3. Run all cells.

## 1. Clone repo

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"  # replace this
BRANCH     = None  # set to a branch name (e.g. "dev") to clone only that branch, or None for the default

!rm -rf /kaggle/working/repo
if BRANCH:
    !git clone --branch {BRANCH} --single-branch {GITHUB_URL} /kaggle/working/repo
else:
    !git clone {GITHUB_URL} /kaggle/working/repo

In [ ]:
import os, sys
os.chdir("/kaggle/working/repo")
# src/ holds both the ptcg/ and cg/ packages
sys.path.insert(0, "/kaggle/working/repo/src")

## 2. Card database

Loads typed card/attack data via `cg.api`.  
**If this fails, `RuleBasedAgent` is identical to `RandomAgent`** — every rule
that needs card data falls back to `RandomFallback`.

In [ ]:
from ptcg import card_db

loaded = card_db.load()
print(f"card_db loaded : {loaded}")

if loaded:
    print(f"Cards          : {len(card_db._cards)}")
    print(f"Attacks        : {len(card_db._attacks)}")
    sample = card_db.get_card(721)
    if sample:
        print(f"Sample (721)   : {sample.name}  "
              f"HP={sample.hp}  "
              f"type={sample.energyType}  "
              f"weakness={sample.weakness}")
else:
    print("WARNING: card_db failed to load. RuleBasedAgent will behave identically to RandomAgent.")

## 3. Available decks

All named decks defined in `ptcg/decks.py`. Pick which deck each agent plays in
section 4 via `DECK_CHOICE`.

In [ ]:
from collections import Counter
from ptcg.decks import DECKS
from ptcg.rules.schema import CardType


def summarize_deck(name: str, deck: list[int]) -> None:
    print(f"{name}  ({len(deck)} cards)")
    counts = Counter(deck)
    pokemon, others = [], 0
    for cid, n in counts.items():
        card = card_db.get_card(cid) if loaded else None
        if card and card.cardType == CardType.POKEMON:
            pokemon.append((n, card.name))
        else:
            others += n
    for n, nm in sorted(pokemon, key=lambda x: -x[0]):
        print(f"    {n}x {nm}")
    print(f"    + {others} trainer / energy cards")
    print()


for deck_name, deck_list in DECKS.items():
    summarize_deck(deck_name, deck_list)

## 4. Configure tournament

- `AGENTS` — agent class names to include.
- `NICKNAMES` — display name per agent (shown in logs/replays).
- `DECK_CHOICE` — which named deck each agent plays (key into `DECKS`).
- `N_GAMES` — games per matchup.

When two of the **same** agent play each other, the display names are
automatically suffixed `(P0)` / `(P1)` so you can tell the two sides apart.

In [ ]:
AGENTS = [
    "RandomAgent",
    "RuleBasedAgent",
    "PlannerAgent",
]

NICKNAMES = {
    "RandomAgent":    "Rookie",
    "RuleBasedAgent": "Rule Master",
    "PlannerAgent":   "Strategist",
}

DECK_CHOICE = {
    "RandomAgent":    "default",
    "RuleBasedAgent": "default",
    "PlannerAgent":   "mega_lucario_ex",
}

N_GAMES = 20


def nickname(agent_class: str) -> str:
    return NICKNAMES.get(agent_class, agent_class)


def agent_deck(agent_class: str) -> list[int]:
    return DECKS[DECK_CHOICE.get(agent_class, "default")]


def display_names(class0: str, class1: str) -> list[str]:
    """Two distinct labels; disambiguate when both sides are the same class."""
    n0, n1 = nickname(class0), nickname(class1)
    if n0 == n1:
        return [f"{n0} (P0)", f"{n1} (P1)"]
    return [n0, n1]

## 5. Run tournament

In [ ]:
import importlib, re
from collections import defaultdict
from kaggle_environments import make


def load_agent(class_name: str, deck: list[int]):
    module_name = re.sub(r"(?<!^)(?=[A-Z])", "_", class_name).lower()
    module = importlib.import_module(f"ptcg.agents.{module_name}")
    return getattr(module, class_name)(deck=deck)


def run_battle(agent0, agent1):
    env = make("cabt", configuration={"decks": [agent0.get_deck(), agent1.get_deck()]})
    env.run([agent0, agent1])
    rewards = [step["reward"] for step in env.steps[-1]]
    return rewards, env


records       = defaultdict(lambda: {"wins": 0, "draws": 0, "losses": 0})
rule_logs     = defaultdict(list)
matchup_envs  = {}   # (class0, class1) -> env of the last game of that matchup
matchup_names = {}   # (class0, class1) -> [display0, display1]

for name0 in AGENTS:
    for name1 in AGENTS:
        key   = (name0, name1)
        names = display_names(name0, name1)
        matchup_names[key] = names
        for _ in range(N_GAMES):
            a0 = load_agent(name0, agent_deck(name0))
            a1 = load_agent(name1, agent_deck(name1))
            a0.on_game_start()
            a1.on_game_start()
            rewards, env = run_battle(a0, a1)
            matchup_envs[key] = env
            result = {"steps": env.steps, "rewards": rewards}
            a0.on_game_end(result)
            a1.on_game_end(result)
            if hasattr(a0, "rule_log"):  rule_logs[name0].extend(a0.rule_log)
            if hasattr(a1, "rule_log"):  rule_logs[name1].extend(a1.rule_log)
            r0 = rewards[0]
            if   r0 == 1:  records[key]["wins"]   += 1
            elif r0 == 0:  records[key]["draws"]  += 1
            else:          records[key]["losses"] += 1

        w, d, l = records[key]["wins"], records[key]["draws"], records[key]["losses"]
        print(f"{names[0]} vs {names[1]:<18}  {w}W {d}D {l}L  ({w/N_GAMES*100:.0f}% win rate for {names[0]})")

## 6. Results summary

In [ ]:
print(f"{'Agent 0':<18} {'Agent 1':<18} {'W':>4} {'D':>4} {'L':>4} {'Win%':>6}")
print("-" * 62)
for (name0, name1), r in records.items():
    n0, n1 = matchup_names[(name0, name1)]
    w, d, l = r["wins"], r["draws"], r["losses"]
    total   = w + d + l
    rate    = w / total * 100 if total > 0 else 0
    print(f"{n0:<18} {n1:<18} {w:>4} {d:>4} {l:>4} {rate:>5.0f}%")

## 7. Rule firing breakdown

If `RandomFallback` is near 100%, `card_db` failed to load — check section 2.

In [ ]:
from collections import Counter

for agent_name, log in rule_logs.items():
    if not log:
        continue
    total  = len(log)
    counts = Counter(log)
    print(f"\n{nickname(agent_name)}  ({total} decisions across all games)")
    print("─" * 55)
    for rule_name, count in counts.most_common():
        pct = count / total * 100
        bar = "█" * int(pct / 3)
        print(f"  {rule_name:<25} {count:>5}  ({pct:>5.1f}%)  {bar}")

## 8. Replay a tournament game

Set `REPLAY` to any matchup played in section 5 (any pair from `AGENTS`).

In [ ]:
REPLAY = ("RandomAgent", "RuleBasedAgent")

from ptcg.log_formatter import format_history

replay_env = matchup_envs.get(REPLAY)
if replay_env is None:
    print(f"No game recorded for {REPLAY}.")
    print(f"Available matchups: {list(matchup_envs.keys())}")
else:
    history = replay_env.steps[0][0].get("visualize")
    names   = matchup_names[REPLAY]   # already disambiguated (e.g. P0 / P1)
    if history is None:
        print("No visualize data available.")
    else:
        print(format_history(history, agent_names=names))

## 9. Custom game

Run and replay a one-off game between any two agents, with **nicknames and decks
set at runtime** here. Set the same class on both sides with different nicknames
(e.g. "Alice" vs "Bob") to follow a mirror match clearly. Does not affect the
tournament records above.

In [ ]:
# ── configure at runtime ──────────────────────────────────────────────────────
CUSTOM_AGENT_0    = "PlannerAgent"
CUSTOM_AGENT_1    = "RandomAgent"
CUSTOM_NICKNAME_0 = "Strategist"
CUSTOM_NICKNAME_1 = "Rookie"
CUSTOM_DECK_0     = "mega_lucario_ex"   # key into DECKS
CUSTOM_DECK_1     = "default"
# ──────────────────────────────────────────────────────────────────────────────

from ptcg.log_formatter import format_history

ca0 = load_agent(CUSTOM_AGENT_0, DECKS[CUSTOM_DECK_0])
ca1 = load_agent(CUSTOM_AGENT_1, DECKS[CUSTOM_DECK_1])
ca0.on_game_start()
ca1.on_game_start()
custom_rewards, custom_env = run_battle(ca0, ca1)
ca0.on_game_end({"rewards": custom_rewards})
ca1.on_game_end({"rewards": custom_rewards})

r0, r1 = custom_rewards
winner = CUSTOM_NICKNAME_0 if r0 == 1 else (CUSTOM_NICKNAME_1 if r1 == 1 else "Draw")
print(f"{CUSTOM_NICKNAME_0} ({CUSTOM_DECK_0}) vs {CUSTOM_NICKNAME_1} ({CUSTOM_DECK_1})  →  winner: {winner}\n")

history = custom_env.steps[0][0].get("visualize")
if history is None:
    print("No visualize data available.")
else:
    print(format_history(history, agent_names=[CUSTOM_NICKNAME_0, CUSTOM_NICKNAME_1]))